# 01_02 — Transfer Acini Segmentation to AnnData

Reads the per-sample acini segmentation CSVs produced by `01_01_acini_segmentation.py`
and merges them into a single AnnData object.

**Inputs**
- `../data/adata.h5ad` — cell-level AnnData (one row per cell)
- `../results/dbscan_epi_dapi_cell/*.csv` — per-sample acini assignments from 01_01

**Output**
- `../data/adata_acini.h5ad` — AnnData with five new `obs` columns:
  - `dapi_mask` — 1 = in DAPI-bright region, -1 = background
  - `ACTA2_mask` — 1 = ACTA2 high (myoepithelial marker)
  - `EPCAM_mask` — 1 = EPCAM high (luminal epithelial marker)
  - `acini_luminal` — DBSCAN cluster label (-1 = unassigned)
  - `acini_afterdil` — cluster label after 1-cell-diameter gap-filling dilation


In [1]:
import glob
import os
import pandas as pd
import anndata
import scanpy as sc

In [2]:
# Collect per-sample acini CSV files produced by 01_01
fnlist = glob.glob('../results/dbscan_epi_dapi_cell/*csv')
print(f"Found {len(fnlist)} sample files.")

# Load the full cell-level AnnData
f_adata = '../data/adata.h5ad'
adata = sc.read(f_adata)
print(adata)

Found 1 sample files.
AnnData object with n_obs × n_vars = 20981 × 380
    obs: 'Batch', 'x_centroid', 'y_centroid', 'x_centroid_px', 'y_centroid_px'
    var: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'


In [3]:
# Initialise acini columns with -1 (unassigned)
for col in ['dapi_mask', 'ACTA2_mask', 'EPCAM_mask', 'acini_luminal', 'acini_afterdil']:
    adata.obs[col] = -1

# Transfer per-sample segmentation results into the merged AnnData
for fn in fnlist:
    df_db = pd.read_csv(fn, index_col=0)
    for col in ['dapi_mask', 'ACTA2_mask', 'EPCAM_mask', 'acini_luminal', 'acini_afterdil']:
        adata.obs.loc[df_db.index, col] = df_db[col]

print("Transfer complete.")

Transfer complete.


In [4]:
# Save annotated AnnData
adata.write('../data/adata_acini.h5ad')
print("Saved to ../data/adata_acini.h5ad")

Saved to ../data/adata_acini.h5ad
